# 01 Ingest Bronze

This notebook implements a local Bronze ingestion pipeline.

The Bronze layer preserves raw records and adds ingestion metadata. Files are ingested from a local landing zone, written to Bronze, logged, and then moved to archive.

Pipeline flow:

```text
landing/ -> bronze/ -> archive/
              |
              v
        checkpoints/ingested_files.json
```


## 1. Setup

Define paths, tables, primary keys, and initialize Spark.


In [39]:
from pathlib import Path
import json
import shutil
from datetime import datetime, timezone

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, current_timestamp, lit
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
)

In [28]:
spark = (
    SparkSession.builder
    .appName("supply-chain-data-platform-bronze")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

26/06/04 13:09:37 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [29]:
DATA_DIR = Path("../data")

LANDING_DIR = DATA_DIR / "landing"
BRONZE_DIR = DATA_DIR / "bronze"
ARCHIVE_DIR = DATA_DIR / "archive"
CHECKPOINT_DIR = DATA_DIR / "checkpoints"

INGESTION_LOG_PATH = CHECKPOINT_DIR / "ingested_files.json"

TABLES = ["orders", "customers", "products", "regions"]

PRIMARY_KEYS = {
    "orders": "order_id",
    "customers": "customer_id",
    "products": "product_id",
    "regions": "region_id",
}

In [30]:
for base_dir in [LANDING_DIR, BRONZE_DIR, ARCHIVE_DIR]:
    for table in TABLES:
        (base_dir / table).mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Ingestion log helpers

The ingestion log prevents duplicate file ingestion. A file is considered already processed only if the combination of `source_table + file_name` has a previous `SUCCESS` status.


In [31]:
def current_utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat()


def load_ingestion_log() -> list[dict]:
    if not INGESTION_LOG_PATH.exists():
        return []

    with open(INGESTION_LOG_PATH, "r") as f:
        return json.load(f)


def save_ingestion_log(log_records: list[dict]) -> None:
    with open(INGESTION_LOG_PATH, "w") as f:
        json.dump(log_records, f, indent=2)


def append_log_record(record: dict) -> None:
    log_records = load_ingestion_log()
    log_records.append(record)
    save_ingestion_log(log_records)


def already_ingested(source_table: str, file_name: str, log_records: list[dict]) -> bool:
    return any(
        record.get("source_table") == source_table
        and record.get("file_name") == file_name
        and record.get("status") == "SUCCESS"
        for record in log_records
    )

## 3. Discover new landing files

The landing zone represents unprocessed incoming files. The folder name determines the target Bronze table.


In [32]:
def discover_landing_files() -> list[dict]:
    discovered_files = []

    for table in TABLES:
        table_landing_dir = LANDING_DIR / table

        for file_path in sorted(table_landing_dir.glob("*.parquet")):
            file_stat = file_path.stat()

            discovered_files.append({
                "source_table": table,
                "file_name": file_path.name,
                "file_path": file_path,
                "file_size_bytes": file_stat.st_size,
                "modified_time": datetime.fromtimestamp(
                    file_stat.st_mtime,
                    tz=timezone.utc,
                ).isoformat(),
            })

    return discovered_files


def get_new_landing_files() -> list[dict]:
    log_records = load_ingestion_log()
    landing_files = discover_landing_files()

    return [
        file_info
        for file_info in landing_files
        if not already_ingested(
            source_table=file_info["source_table"],
            file_name=file_info["file_name"],
            log_records=log_records,
        )
    ]

## 4. Bronze ingestion functions

Bronze does not clean, join, or deduplicate business records. It only preserves raw data, adds metadata, and logs ingestion results.

Duplicate primary keys are detected and logged, but not removed in Bronze. Deduplication belongs in Silver.


In [33]:
def count_duplicate_keys(df, key_column: str) -> int:
    return (
        df
        .groupBy(key_column)
        .agg(count("*").alias("n"))
        .filter(col("n") > 1)
        .count()
    )

In [34]:
def ingest_single_file(file_info: dict) -> None:
    source_table = file_info["source_table"]
    file_name = file_info["file_name"]
    file_path = file_info["file_path"]

    bronze_path = BRONZE_DIR / source_table
    archive_path = ARCHIVE_DIR / source_table / file_name

    log_records = load_ingestion_log()

    if already_ingested(source_table, file_name, log_records):
        print(f"Skipping already ingested file: {source_table}/{file_name}")
        return

    append_log_record({
        "file_name": file_name,
        "source_table": source_table,
        "landing_path": str(file_path),
        "archive_path": str(archive_path),
        "file_size_bytes": file_info["file_size_bytes"],
        "modified_time": file_info["modified_time"],
        "ingested_at": current_utc_timestamp(),
        "rows_ingested": None,
        "duplicate_key_count": None,
        "status": "STARTED",
        "error_message": None,
    })

    try:
        df = spark.read.parquet(str(file_path))

        rows_ingested = df.count()
        duplicate_key_count = count_duplicate_keys(
            df=df,
            key_column=PRIMARY_KEYS[source_table],
        )

        df_with_metadata = (
            df
            .withColumn("source_file", lit(file_name))
            .withColumn("source_table", lit(source_table))
            .withColumn("source_file_size_bytes", lit(file_info["file_size_bytes"]))
            .withColumn("source_file_modified_time", lit(file_info["modified_time"]))
            .withColumn("ingestion_timestamp", current_timestamp())
        )

        (
            df_with_metadata
            .write
            .mode("append")
            .parquet(str(bronze_path))
        )

        archive_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(file_path), str(archive_path))

        append_log_record({
            "file_name": file_name,
            "source_table": source_table,
            "landing_path": str(file_path),
            "archive_path": str(archive_path),
            "file_size_bytes": file_info["file_size_bytes"],
            "modified_time": file_info["modified_time"],
            "ingested_at": current_utc_timestamp(),
            "rows_ingested": rows_ingested,
            "duplicate_key_count": duplicate_key_count,
            "status": "SUCCESS",
            "error_message": None,
        })

        print(
            f"Ingested {source_table}/{file_name}: "
            f"{rows_ingested} rows, "
            f"{duplicate_key_count} duplicate key(s)."
        )

    except Exception as error:
        append_log_record({
            "file_name": file_name,
            "source_table": source_table,
            "landing_path": str(file_path),
            "archive_path": str(archive_path),
            "file_size_bytes": file_info["file_size_bytes"],
            "modified_time": file_info["modified_time"],
            "ingested_at": current_utc_timestamp(),
            "rows_ingested": None,
            "duplicate_key_count": None,
            "status": "FAILED",
            "error_message": str(error),
        })

        print(f"Failed to ingest {source_table}/{file_name}: {error}")

In [35]:
def run_bronze_ingestion() -> None:
    new_files = get_new_landing_files()

    if not new_files:
        print("No new files to ingest.")
        return

    print(f"Found {len(new_files)} new file(s) to ingest.")

    for file_info in new_files:
        ingest_single_file(file_info)

## 5. Run Bronze ingestion

Place new `.parquet` files in `data/landing/<table_name>/`, then run this cell.

After successful ingestion, files are moved to `data/archive/<table_name>/`.


In [36]:
run_bronze_ingestion()

No new files to ingest.


## 6. Validate Bronze outputs

This section checks row counts, source-file counts, ingestion log status, and remaining files in landing.


In [37]:
def parquet_data_exists(path: Path) -> bool:
    return path.exists() and any(path.rglob("*.parquet"))


for table in TABLES:
    bronze_path = BRONZE_DIR / table

    print("=" * 80)
    print(f"Bronze table: {table}")

    if parquet_data_exists(bronze_path):
        df = spark.read.parquet(str(bronze_path))
        print(f"Rows: {df.count()}")
        df.groupBy("source_file", "source_table").count().show(truncate=False)
    else:
        print("No bronze data found.")

Bronze table: orders
Rows: 9990
+--------------------+------------+-----+
|source_file         |source_table|count|
+--------------------+------------+-----+
|orders_first.parquet|orders      |9990 |
+--------------------+------------+-----+

Bronze table: customers
Rows: 1990
+----------------------+------------+-----+
|source_file           |source_table|count|
+----------------------+------------+-----+
|customer_first.parquet|customers   |1990 |
+----------------------+------------+-----+

Bronze table: products
Rows: 490
+----------------------+------------+-----+
|source_file           |source_table|count|
+----------------------+------------+-----+
|products_first.parquet|products    |490  |
+----------------------+------------+-----+

Bronze table: regions
Rows: 4
+---------------+------------+-----+
|source_file    |source_table|count|
+---------------+------------+-----+
|regions.parquet|regions     |4    |
+---------------+------------+-----+



In [40]:

ingestion_log_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("source_table", StringType(), True),
    StructField("landing_path", StringType(), True),
    StructField("archive_path", StringType(), True),
    StructField("file_size_bytes", LongType(), True),
    StructField("modified_time", StringType(), True),
    StructField("ingested_at", StringType(), True),
    StructField("rows_ingested", LongType(), True),
    StructField("duplicate_key_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

In [41]:
log_records = load_ingestion_log()

if log_records:
    log_df = spark.createDataFrame(
        log_records,
        schema=ingestion_log_schema,
    )

    log_df.groupBy("status").count().show()

    log_df.select(
        "source_table",
        "file_name",
        "status",
        "rows_ingested",
        "duplicate_key_count",
        "ingested_at",
    ).orderBy(
        "source_table",
        "file_name",
        "ingested_at",
    ).show(truncate=False)
else:
    print("No ingestion log records found.")

+-------+-----+
| status|count|
+-------+-----+
|SUCCESS|    4|
+-------+-----+

+------------+----------------------+-------+-------------+-------------------+--------------------------------+
|source_table|file_name             |status |rows_ingested|duplicate_key_count|ingested_at                     |
+------------+----------------------+-------+-------------+-------------------+--------------------------------+
|customers   |customer_first.parquet|SUCCESS|1990         |0                  |2026-06-04T11:02:33.641227+00:00|
|orders      |orders_first.parquet  |SUCCESS|9990         |0                  |2026-06-04T11:00:03.732480+00:00|
|products    |products_first.parquet|SUCCESS|490          |0                  |2026-06-04T11:02:33.991637+00:00|
|regions     |regions.parquet       |SUCCESS|4            |0                  |2026-06-04T11:02:34.279232+00:00|
+------------+----------------------+-------+-------------+-------------------+--------------------------------+



In [42]:
print("Remaining files in landing:")

for table in TABLES:
    files = sorted((LANDING_DIR / table).glob("*.parquet"))
    print(table, [file.name for file in files])

Remaining files in landing:
orders []
customers []
products []
regions []


## 7. Next step

To simulate a second ingestion batch, move these files into the landing zone:

```text
orders_second.parquet    -> data/landing/orders/
customers_second.parquet -> data/landing/customers/
products_second.parquet  -> data/landing/products/
```

Then rerun the notebook from top to bottom.

Expected final Bronze row counts after first and second batches:

```text
orders: 10000
customers: 2000
products: 500
regions: 4
```
